# Практическое занятие 6. Получение данных через API и их сохранение в виде DataFrame библиотеки Pandas

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mkb40-svg/big_data_6semestr/blob/main/notebooks/p06.ipynb)

**Цели занятия:**

- Понять, как устроены REST API: адрес ресурса, метод GET, код ответа, JSON в теле ответа
- Научиться отправлять запросы библиотекой `requests`, обходить постраничную выдачу (пагинацию) и обрабатывать ошибки
- Преобразовывать ответы API во вложенном JSON в DataFrame (`pd.json_normalize`) и сохранять их в CSV и JSON

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/mkb40-svg/big_data_6semestr.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Что такое API

**API** (Application Programming Interface) — способ, которым программа получает данные у другой программы. Веб-API работают поверх HTTP: клиент отправляет **запрос** на адрес (URL), сервер возвращает **ответ** — обычно в формате JSON. Так собирают данные из соцсетей, погодных сервисов, онлайн-библиотек, курсов валют ЦБ и т.д.

В тренажёре есть учебный API библиотеки «Talaba Kutubxona». Его адрес хранится в переменной `API`:

| Ресурс | Что возвращает |
|---|---|
| `API + "books/page-1.json"` … `page-4.json` | книги порциями по 5 (поля `page`, `total`, `next`, `results`) |
| `API + "books/7.json"` | одна книга с отзывами (`reviews`) |
| `API + "authors.json"` | авторы и номера их книг |
| `API + "weather/tashkent.json"` | погода за март: также `samarkand`, `bukhara`, `nukus` |

## Запрос с requests

```python
import requests

r = requests.get(API + "books/page-1.json")
r.status_code                  # 200 — успех
r.headers["Content-Type"]      # application/json
r.encoding = "utf-8"           # кодировка текста ответа — задаём явно!
print(r.text[:200])            # тело ответа как строка
data = r.json()                # тело ответа как dict/list Python
```

> **Внимание: Кодировка ответа**  
> Если сервер не указал `charset` в заголовке `Content-Type`, `requests` может решить, что текст в кодировке ISO-8859-1, и узбекские буквы `oʻ, gʻ` или кириллица превратятся в «кракозябры». Поэтому перед `r.text` всегда пишите `r.encoding = "utf-8"`. Метод `r.json()` сам определяет UTF-8.

### Параметры и заголовки запроса

Настоящие API принимают **параметры** в строке запроса (`?city=Toshkent&page=2`) и часто требуют **ключ** в заголовке. В `requests` их не склеивают вручную, а передают словарями — библиотека сама закодирует пробелы, апострофы и кириллицу:

```python
r = requests.get("https://example.org/api/search",
                 params={"q": "Navoiy", "lang": "uz", "page": 2},     # -> ?q=Navoiy&lang=uz&page=2
                 headers={"Authorization": "Bearer <ваш_токен>"},
                 timeout=10)
r.url                          # итоговый адрес с параметрами
```

## Коды ответа HTTP

| Код | Значение |
|---|---|
| **200** OK | всё хорошо |
| **301/302** | ресурс переехал (requests сам перейдёт по новому адресу) |
| **400** Bad Request | неверные параметры запроса |
| **401/403** | нужна авторизация (API-ключ) / доступ запрещён |
| **404** Not Found | ресурса нет |
| **429** Too Many Requests | превышен лимит запросов — нужно подождать |
| **500** и выше | ошибка на сервере |

```python
r = requests.get(API + "books/99.json")
r.status_code              # 404
r.ok                       # False (True для кодов < 400)
try:
    r.raise_for_status()   # бросает исключение для кодов 4xx и 5xx
except requests.HTTPError as e:
    print("Ошибка:", e)
```

## Пагинация

Большие наборы API отдаёт **страницами**. Ответ содержит ссылку на следующую страницу (`next`); на последней странице `next` равен `null` (в Python — `None`).

```python
url = API + "books/page-1.json"
items = []
while url:                                   # пока есть следующая страница
    data = requests.get(url).json()
    items.extend(data["results"])
    url = API + "books/" + data["next"] if data["next"] else None
```

## Из JSON в DataFrame

```python
import pandas as pd

books = pd.json_normalize(items)             # publisher.name, price.amount, ...
d = requests.get(API + "weather/nukus.json").json()
nukus = pd.json_normalize(d, record_path="daily", meta=["city"])   # строка на каждый день

books.to_csv("books.csv", index=False)
books.to_json("books.json", orient="records", force_ascii=False, indent=2)
```

> **Совет: Вежливость к API**  
> Настоящие API ограничивают число запросов. Делайте паузы (`time.sleep(1)`), сохраняйте полученные данные в файл, чтобы не скачивать их повторно, и передавайте `timeout=10`, чтобы программа не «зависла» при сбое сети.

## Задание 1. Первый запрос к API

Отправьте GET-запрос к первой странице каталога книг `API + "books/page-1.json"`.

- `r` — объект ответа; `status` — код ответа; `ctype` — значение заголовка `Content-Type`;
- задайте `r.encoding = "utf-8"` и выведите первые 200 символов текста ответа;
- `data` — ответ, преобразованный в словарь (`r.json()`);
- `n_results` — сколько книг на этой странице, `total` — сколько книг всего (поле `total`), `next_page` — значение поля `next`.

<details><summary>💡 Подсказка</summary>

`r = requests.get(API + "books/page-1.json")`, заголовок: `r.headers["Content-Type"]`, книги на странице: `len(data["results"])`.

</details>

In [ ]:
import requests

r = ...
status = ...
ctype = ...

r.encoding = "utf-8"
print(r.text[:200])

data = ...
n_results = ...
total = ...
next_page = ...

print(status, ctype)
print(f"На странице {n_results} книг из {total}; дальше: {next_page}")

## Задание 2. Пагинация: все страницы

Соберите **все** книги каталога, переходя по ссылке `next`, пока она не станет `None`:

- `all_books` — список словарей всех книг со всех страниц;
- `pages` — сколько страниц было загружено.

Адрес следующей страницы: `API + "books/" + data["next"]`.

<details><summary>💡 Подсказка</summary>

`all_books.extend(data["results"])`; `pages += 1`; `url = API + "books/" + data["next"] if data["next"] else None`.

</details>

In [ ]:
import requests

url = API + "books/page-1.json"
all_books = []
pages = 0

while url:
    r = requests.get(url)
    data = r.json()
    # 1) добавьте книги data["results"] в all_books
    # 2) увеличьте счётчик pages
    # 3) вычислите адрес следующей страницы (или None, если next пуст)
    url = None   # замените эту строку

print("Страниц:", pages, "книг:", len(all_books))
print([b["title"] for b in all_books])

## Задание 3. Вложенный JSON → DataFrame

Список `all_books` (16 книг из API) уже загружен. Постройте удобную таблицу `books`:

1. Примените `pd.json_normalize(all_books)`.
2. Добавьте столбец `author` — имена всех авторов книги через `", "` (в исходных данных `authors` — список словарей), затем удалите столбец `authors`.
3. Превратите `genres` (список) в строку через `", "`.
4. Переименуйте `publisher.name` → `publisher`, `price.amount` → `price`.

Затем посчитайте:
- `by_lang` — Series: средняя цена по языку;
- `n_no_isbn` — у скольких книг нет ISBN (поле необязательное — в таблице будет пропуск).

<details><summary>💡 Подсказка</summary>

Авторы: `books["authors"].apply(lambda lst: ", ".join(a["name"] for a in lst))`. Жанры: `books["genres"].str.join(", ")`. Пропуски: `books["isbn"].isna().sum()`.

</details>

In [ ]:
# Подготовительный код задания
import requests
import pandas as pd
all_books, url = [], API + "books/page-1.json"
while url:
    _d = requests.get(url).json()
    all_books.extend(_d["results"])
    url = API + "books/" + _d["next"] if _d["next"] else None
del _d

In [ ]:
books = ...

books["author"] = ...
books = books.drop(columns=["authors"])
books["genres"] = ...
books = books.rename(columns={...})

by_lang = ...
n_no_isbn = ...

print(by_lang)
print("Без ISBN:", n_no_isbn)
books[["id", "title", "author", "genres", "publisher", "price", "language"]]

## Задание 4. Запросы к отдельным ресурсам: отзывы

Списки страниц не содержат отзывов — они есть только в карточке книги `API + "books/<id>.json"`.

1. Загрузите карточки всех книг с id от 1 до 16 в список `details` (16 запросов).
2. `reviews` — DataFrame: одна строка на отзыв, со столбцами `user`, `stars`, `text`, а также `id` и `title` книги (`json_normalize` с `record_path` и `meta`).
3. `n_reviews` — общее число отзывов; `most_reviewed` — название книги с наибольшим числом отзывов.
4. `no_reviews` — список названий книг, у которых отзывов нет (их не будет в `reviews` — сравните с `details`).

<details><summary>💡 Подсказка</summary>

В цикле: `details.append(requests.get(API + f"books/{book_id}.json").json())`. Таблица: `pd.json_normalize(details, record_path="reviews", meta=["id", "title"])`. Без отзывов: `[b["title"] for b in details if not b["reviews"]]`.

</details>

In [ ]:
import requests
import pandas as pd

details = []
for book_id in range(1, 17):
    ...

reviews = ...
n_reviews = ...
most_reviewed = ...
no_reviews = ...

print("Отзывов:", n_reviews, "| больше всех:", most_reviewed, "| без отзывов:", no_reviews)
reviews.head()

## Задание 5. Погода в 4 городах → один DataFrame

Эндпоинты `API + "weather/<город>.json"` для `tashkent`, `samarkand`, `bukhara`, `nukus` возвращают погоду за март 2026 (массив `daily` и поле `city`).

1. `weather` — **одна** таблица для всех четырёх городов: строка на каждый день каждого города, со столбцом `city` (`json_normalize` + `pd.concat`).
2. Преобразуйте `date` в datetime и добавьте `t_mean = (t_min + t_max) / 2`.
3. `summary` — DataFrame с индексом `city` и столбцами `t_mean` (средняя температура) и `rain` (сумма `precipitation_mm`).
4. `warmest` — самый тёплый город (по средней `t_mean`).
5. Сохраните `weather` в `weather_march.csv` (без индекса) и `weather_march.json` (`orient="records"`, `force_ascii=False`, `date_format="iso"`).

<details><summary>💡 Подсказка</summary>

В цикле: `d = requests.get(API + f"weather/{c}.json").json()` и `frames.append(pd.json_normalize(d, record_path="daily", meta=["city"]))`. Сводка: `weather.groupby("city").agg(t_mean=("t_mean", "mean"), rain=("precipitation_mm", "sum"))`.

</details>

In [ ]:
import requests
import pandas as pd

cities = ["tashkent", "samarkand", "bukhara", "nukus"]
frames = []
for c in cities:
    ...

weather = ...
weather["date"] = ...
weather["t_mean"] = ...

summary = ...
warmest = ...

# сохраните weather в CSV и JSON

print(summary.round(1))
print("Самый тёплый город:", warmest)

## Задание 6. Коды ответа и обработка ошибок

Не каждый запрос успешен. Напишите надёжную функцию загрузки.

1. `fetch_json(url)` — делает GET-запрос; если код ответа ошибочный (4xx/5xx) или произошла сетевая ошибка, печатает сообщение и возвращает `None`, иначе возвращает `r.json()`. Используйте `raise_for_status()` и `try/except requests.exceptions.RequestException`.
2. `codes` — словарь `{эндпоинт: код ответа}` для всех эндпоинтов из списка `endpoints`.
3. `loaded` — сколько эндпоинтов из списка функция `fetch_json` загрузила успешно (вернула не `None`).

<details><summary>💡 Подсказка</summary>

Шаблон: `try: r = requests.get(url, timeout=10); r.raise_for_status(); return r.json()` и `except requests.exceptions.RequestException as e: print("Ошибка:", e); return None`. Подсчёт: `sum(fetch_json(API + ep) is not None for ep in endpoints)`.

</details>

In [ ]:
import requests

endpoints = ["books/1.json", "books/99.json", "authors.json",
             "weather/tashkent.json", "weather/andijan.json", "books/page-5.json"]

def fetch_json(url):
    r = requests.get(url, timeout=10)
    return r.json()

codes = {}
for ep in endpoints:
    ...

loaded = ...

print(codes)
print("Загружено успешно:", loaded, "из", len(endpoints))

## Самостоятельная работа

## Самостоятельная работа: «Изучение возможностей библиотеки Pandas»

1. Загрузите `API + "authors.json"` и объедините его с таблицей книг (`books` из задания 3) так, чтобы у каждой книги появилась страна автора (`explode` по списку `books` автора и `merge` по id).
2. Через Open-Meteo в Colab получите прогноз на неделю для Ташкента, Самарканда, Бухары и Нукуса одним циклом и постройте линейный график максимальной температуры.
3. Напишите функцию `fetch_all(first_url)`, которая обходит любую пагинацию с полем `next` и возвращает DataFrame; добавьте паузу `time.sleep(0.5)` между запросами и повтор запроса (до 3 раз) при коде 429 или 5xx.